# B2-022-probabilistic-latent-models — Practice p15 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** kl-divergence

*45 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:F6-svd-spectral`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C9-dimensionality-reduction`, `aio-ml:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:F6-svd-spectral](../../../../aio-ml/units/12-F6-svd-spectral/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C9-dimensionality-reduction](../../../../aio-ml/units/16-C9-dimensionality-reduction/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

### (a) Log-ratio

$\log p(x)=-\tfrac12\log(2\pi\sigma^2)-\frac{(x-\mu)^2}{2\sigma^2}$ and $\log q(x)=-\tfrac12\log(2\pi)-\frac{x^2}{2}$. Therefore
$$\log p(x)-\log q(x)=-\tfrac12\log\sigma^2-\frac{(x-\mu)^2}{2\sigma^2}+\frac{x^2}{2}.$$

### (b) Expectations term by term

With $x\sim p=N(\mu,\sigma^2)$ (Book 1: $E[x]=\mu$, $\operatorname{Var}(x)=\sigma^2$, and linearity of expectation):

- $E[(x-\mu)^2]=\operatorname{Var}(x)=\sigma^2$ — the definition of variance.
- $E[x^2]=\operatorname{Var}(x)+(E[x])^2=\sigma^2+\mu^2$ — from $\operatorname{Var}(x)=E[x^2]-(E[x])^2$.
- The constant $-\tfrac12\log\sigma^2$ has expectation itself.

So $\mathrm{KL}(p\|q)=-\tfrac12\log\sigma^2-\frac{\sigma^2}{2\sigma^2}+\frac{\sigma^2+\mu^2}{2}$.

### (c) Closed form

$$\mathrm{KL}(p\|q)=\tfrac12\big(\mu^2+\sigma^2-\log\sigma^2-1\big),$$
and with $\ell=\log\sigma^2$ (so $\sigma^2=e^\ell$): $\mathrm{KL}=\tfrac12\big(\mu^2+e^\ell-\ell-1\big)$.

### (d) Nonnegativity and equality

The lemma $\log x\le x-1$ with $x=\sigma^2>0$ gives $\sigma^2-\log\sigma^2-1\ge0$, with equality only at $\sigma^2=1$. Also $\mu^2\ge0$ with equality only at $\mu=0$. The KL is half the sum of these two nonnegative quantities, so $\mathrm{KL}\ge0$, and it is $0$ iff both are $0$, i.e. iff $\mu=0$ and $\sigma^2=1$.

### (e) Evaluation at $\mu=1$, $\ell=1$

$\mathrm{KL}=\tfrac12(1+e-1-1)=\tfrac{e-1}{2}\approx0.859141$.

## Numeric sanity check

In [ ]:
import math
import torch

def closed_form(mu, ell):
    return 0.5 * (mu ** 2 + math.exp(ell) - ell - 1)  # math.exp: e^x

def mc_kl(mu, ell, n=400_000, seed=20260927):
    g = torch.Generator().manual_seed(seed)
    sigma = math.exp(0.5 * ell)
    x = mu + sigma * torch.randn(n, generator=g, dtype=torch.float64)
    log_p = -0.5 * math.log(2 * math.pi * sigma ** 2) - (x - mu) ** 2 / (2 * sigma ** 2)
    log_q = -0.5 * math.log(2 * math.pi) - x ** 2 / 2
    return (log_p - log_q).mean().item()

value_e = closed_form(1.0, 1.0)
mc_e = mc_kl(1.0, 1.0)
grid = [closed_form(m, l) for m in (-1.0, -0.1, 0.0, 0.3) for l in (-2.0, -0.2, 0.0, 0.7)]
print(round(value_e, 6), mc_e)

### Answer check

In [ ]:
# Checks: the KL-divergence answers against pinned values
assert abs(value_e - (math.e - 1) / 2) < 1e-15
assert round(value_e, 6) == 0.859141
assert abs(mc_e - value_e) < 0.02
assert all(v >= 0 for v in grid)
assert closed_form(0.0, 0.0) == 0.0